In [1]:
#Task 1  —  Design a Class
class TrainingRun:
    run_count = 0
    def __init__(self, model_name, learning_rate):
        if not self.validate_learning_rate(learning_rate):
            raise ValueError("Learning rate must be between 0 and 1.")

        self.model_name = model_name
        self.learning_rate = learning_rate
        self.status = "Not Started"
        TrainingRun.run_count += 1
        
    def start(self):
        self.status = "Running"
        print(f"Training started for {self.model_name}")

    def summary(self):
        return (
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

    @classmethod
    def from_config(cls, config_dict):
        return cls(
            config_dict["model_name"],
            config_dict["learning_rate"]
        )

    @staticmethod
    def validate_learning_rate(lr):
        return 0 < lr < 1

run1 = TrainingRun("CNN", 0.01)
run2 = TrainingRun("Transformer", 0.001)

config = {
    "model_name": "RandomForest",
    "learning_rate": 0.1
}
run3 = TrainingRun.from_config(config)

run1.start()

print(run1.summary())
print(run2.summary())
print(run3.summary())

print("Total runs created:", TrainingRun.run_count)

Training started for CNN
Model: CNN, Learning Rate: 0.01, Status: Running
Model: Transformer, Learning Rate: 0.001, Status: Not Started
Model: RandomForest, Learning Rate: 0.1, Status: Not Started
Total runs created: 3


In [2]:
#Task 2  —  Encapsulation
#PILLAR: ENCAPSULATION
#Objective: Apply Python's encapsulation conventions (public / _protected / __private)

class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        self.__api_key = "fake_api_token_123"
        TrainingRun.run_count += 1

    @property
    def status(self):
        return self._status

    @status.setter
    def status(self, value):
        allowed_statuses = ["pending", "running", "done"]

        if value not in allowed_statuses:
            raise ValueError(
                f"Status must be one of {allowed_statuses}"
            )

        self._status = value

    def start(self):
        self.status = "running"

    def summary(self):
        return (
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )



run = TrainingRun("CNN", 0.01)

run.status = "running"
print(run.summary())

try:
    run.status = "paused"
except ValueError as e:
    print("Error:", e)

try:
    print(run.__api_key)
except AttributeError as e:
    print("AttributeError:", e)

print("API Key:", run._TrainingRun__api_key)

Model: CNN, Learning Rate: 0.01, Status: running
Error: Status must be one of ['pending', 'running', 'done']
AttributeError: 'TrainingRun' object has no attribute '__api_key'
API Key: fake_api_token_123


In [3]:
#Task 3  —  Inheritance
# PILLAR: INHERITANCE
# Objective: Implement inheritance and method overriding
class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self.status = "pending"

        TrainingRun.run_count += 1

    def start(self):
        self.status = "running"

    def summary(self):
        return (
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )


class LRSchedulerRun(TrainingRun):
    def __init__(self, model_name, learning_rate, schedule):
        # Reuse parent constructor
        super().__init__(model_name, learning_rate)
        self.schedule = schedule

    def summary(self):
        parent_summary = super().summary()
        current_lr = self.schedule[0] if self.schedule else self.learning_rate

        return (
            f"{parent_summary}, "
            f"Current Scheduled LR: {current_lr}"
        )

run1 = TrainingRun("CNN", 0.01)

scheduler_run = LRSchedulerRun(
    "Transformer",
    0.001,
    [0.001, 0.0005, 0.0001]
)

print(run1.summary())
print(scheduler_run.summary())

print("Total Runs Created:", TrainingRun.run_count)


Model: CNN, Learning Rate: 0.01, Status: pending
Model: Transformer, Learning Rate: 0.001, Status: pending, Current Scheduled LR: 0.001
Total Runs Created: 2


In [4]:
# Task 4  —  Polymorphism
# PILLAR: POLYMORPHISM
# Objective: Use polymorphism to treat different subclasses interchangeably
# Base class
class TrainingRun:
    def __init__(self, model_name, epochs):
        self.model_name = model_name
        self.epochs = epochs

    def summary(self):
        print(
            f"TrainingRun -> Model: {self.model_name}, "
            f"Epochs: {self.epochs}"
        )

class LRSchedulerRun(TrainingRun):
    def __init__(self, model_name, epochs, learning_rate):
        super().__init__(model_name, epochs)
        self.learning_rate = learning_rate

    def summary(self):
        print(
            f"LRSchedulerRun -> Model: {self.model_name}, "
            f"Epochs: {self.epochs}, "
            f"Learning Rate: {self.learning_rate}"
        )

class EarlyStoppingRun(TrainingRun):
    def __init__(self, model_name, epochs, patience):
        super().__init__(model_name, epochs)
        self.patience = patience

    def summary(self):
        print(
            f"EarlyStoppingRun -> Model: {self.model_name}, "
            f"Epochs: {self.epochs}, "
            f"Patience: {self.patience}"
        )


def print_all_summaries(runs):
    for run in runs:
        run.summary()  
runs = [
    TrainingRun("ResNet50", 20),
    LRSchedulerRun("VisionTransformer", 30, 0.001),
    EarlyStoppingRun("BERT", 25, 5)
]

print_all_summaries(runs)

TrainingRun -> Model: ResNet50, Epochs: 20
LRSchedulerRun -> Model: VisionTransformer, Epochs: 30, Learning Rate: 0.001
EarlyStoppingRun -> Model: BERT, Epochs: 25, Patience: 5


In [5]:
# Task 5  —  Duck Typing vs. Interfaces
# PILLAR: CONTRACTS
# Objective: Contrast duck typing / ABCs with Java's interfaces
# Unrelated classes using duck typing

class Cleaner:
    def process(self, data):
        return data.strip()

class Tokenizer:
    def process(self, data):
        return data.split()

class Normalizer:
    def process(self, data):
        return [token.lower() for token in data]

def run_pipeline(steps, data):
    for step in steps:
        data = step.process(data)
    return data

steps = [
    Cleaner(),
    Tokenizer(),
    Normalizer()
]

result = run_pipeline(steps, "  Hello WORLD Python  ")
print(result)


['hello', 'world', 'python']


In [6]:
# Task 6  —  Dunder Methods
# PILLAR: DUNDER METHODS
# Objective: Implement at least one dunder method
class TrainingRun:
    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate

    def __str__(self):
        """
        Human-readable representation.
        Used by print() and str().
        """
        return (
            f"TrainingRun(model='{self.model_name}', "
            f"lr={self.learning_rate})"
        )

    def __repr__(self):
        """
        Developer-focused representation.
        Ideally detailed and unambiguous enough to recreate the object.
        """
        return (
            f"TrainingRun(model_name='{self.model_name}', "
            f"learning_rate={self.learning_rate})"
        )

    def __eq__(self, other):
        """
        Two TrainingRun objects are equal if they have the same
        model_name and learning_rate.
        """
        if not isinstance(other, TrainingRun):
            return NotImplemented

        return (
            self.model_name == other.model_name
            and self.learning_rate == other.learning_rate
        )

run1 = TrainingRun("gpt-mini", 0.01)
run2 = TrainingRun("gpt-mini", 0.01)
run3 = TrainingRun("bert-base", 0.001)

print(run1)
print(run1 == run2)  
print(run1 == run3)  
print(repr(run1))

TrainingRun(model='gpt-mini', lr=0.01)
True
False
TrainingRun(model_name='gpt-mini', learning_rate=0.01)
